In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/eslamfawzysayed/aplication-3-try/final_data_complete.xlsx


# Step 1: Install libraries

In [2]:
# ============================================================
# Step 1: Install libraries
# ============================================================

!pip install pymoo -q
!pip install openpyxl -q
!pip install gradio -q

print("✅ All libraries installed successfully")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.3/328.3 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 4.7 MB/s eta 0:00:00
✅ All libraries installed successfully


# Step 2: Import libraries and load data



In [3]:
# ============================================================
# Step 2: Import libraries and load data
# ============================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize
import joblib
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("🌱 Hydrochar Optimization System - Full Run")
print("=" * 70)

# Load data
file_path = '/kaggle/input/datasets/eslamfawzysayed/aplication-3-try/final_data_complete.xlsx'
df = pd.read_excel(file_path)
print(f"✅ Loaded {len(df)} rows of data")

# Define columns (as mentioned)
input_cols = [
    'C%', 'H%', 'N%', 'S%', 'O%', 'Ash%', 
    'Volatile matter%', 'Fixed carbon%', 
    'HHV (MJ/Kg)', 'DM(%)', 'water Content (%)',
    'Temprature(°C)', 'Reaction Time (min)'
]

output_cols = [
    'yield(%)', 'Cc (%)', 'Hc(%)', 'Nc(%)', 'Sc(%)', 'Oc(%)',
    'HHVc (MJ/Kg)', 'EY(%)', 'ED(%)', 'CR(%)',
    'HC- Volatile matter%', 'HC- Fixed carbon%', 'HC- Ash(%)'
]

# Base columns (feedstock properties only)
base_input_cols = [
    'C%', 'H%', 'N%', 'S%', 'O%', 'Ash%', 
    'Volatile matter%', 'Fixed carbon%', 
    'HHV (MJ/Kg)', 'DM(%)', 'water Content (%)'
]

# Filter existing columns
input_cols = [col for col in input_cols if col in df.columns]
output_cols = [col for col in output_cols if col in df.columns]
base_input_cols = [col for col in base_input_cols if col in df.columns]

print(f"📊 Inputs: {len(input_cols)} (11 properties + 2 conditions)")
print(f"📊 Outputs: {len(output_cols)} properties")
print(f"📊 Feedstock properties: {len(base_input_cols)} properties")

🌱 Hydrochar Optimization System - Full Run
✅ Loaded 271 rows of data
📊 Inputs: 13 (11 properties + 2 conditions)
📊 Outputs: 13 properties
📊 Feedstock properties: 11 properties


# 
# 🎯 # Step 3: Train forward models (for prediction during optimization)

In [4]:
# ============================================================
# Step 3: Train forward models (for prediction during optimization)
# ============================================================

print("\n" + "=" * 70)
print("🔄 Training forward models (13 models)...")
print("=" * 70)

X = df[input_cols]  # 11 properties + temperature + time
Y = df[output_cols]  # 13 outputs

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

models_forward = {}

for i, output_col in enumerate(output_cols):
    print(f"  Training {i+1}/{len(output_cols)}: {output_col}")
    model = RandomForestRegressor(
        n_estimators=50,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, Y_train[output_col])
    models_forward[output_col] = model

print("✅ All forward models trained successfully")

# Min/max values for normalization
output_mins = {col: df[col].min() for col in output_cols}
output_maxs = {col: df[col].max() for col in output_cols}


🔄 Training forward models (13 models)...
  Training 1/13: yield(%)
  Training 2/13: Cc (%)
  Training 3/13: Hc(%)
  Training 4/13: Nc(%)
  Training 5/13: Sc(%)
  Training 6/13: Oc(%)
  Training 7/13: HHVc (MJ/Kg)
  Training 8/13: EY(%)
  Training 9/13: ED(%)
  Training 10/13: CR(%)
  Training 11/13: HC- Volatile matter%
  Training 12/13: HC- Fixed carbon%
  Training 13/13: HC- Ash(%)
✅ All forward models trained successfully


# 📊 # Step 4: Application criteria (according to the table)

In [5]:
# ============================================================
# Step 4: Application criteria (according to the table)
# ============================================================

print("\n" + "=" * 70)
print("📊 Application criteria")
print("=" * 70)

application_criteria = {
    'Energy': {
        'name': 'Energy Production',
        'weights': {
            'yield(%)': 0.15, 'Cc (%)': 0.10, 'Hc(%)': 0.05,
            'Nc(%)': 0.05, 'Sc(%)': 0.05, 'Oc(%)': 0.05,
            'HHVc (MJ/Kg)': 0.25, 'EY(%)': 0.05, 'ED(%)': 0.05,
            'CR(%)': 0.05, 'HC- Volatile matter%': 0.05,
            'HC- Fixed carbon%': 0.10, 'HC- Ash(%)': 0.05
        },
        'targets': {
            'yield(%)': 'max', 'Cc (%)': 'max', 'Hc(%)': 'max',
            'Nc(%)': 'min', 'Sc(%)': 'min', 'Oc(%)': 'min',
            'HHVc (MJ/Kg)': 'max', 'EY(%)': 'max', 'ED(%)': 'max',
            'CR(%)': 'max', 'HC- Volatile matter%': 'min',
            'HC- Fixed carbon%': 'max', 'HC- Ash(%)': 'min'
        }
    },
    'Soil': {
        'name': 'Soil Amendment',
        'weights': {
            'yield(%)': 0.15, 'Nc(%)': 0.25, 'Sc(%)': 0.20,
            'HC- Ash(%)': 0.30, 'HHVc (MJ/Kg)': 0.10
        },
        'targets': {
            'yield(%)': 'max', 'Nc(%)': 'max', 'Sc(%)': 'max',
            'HC- Ash(%)': 'max', 'HHVc (MJ/Kg)': 'min'
        }
    },
    'Adsorbent': {
        'name': 'Adsorbent Material',
        'weights': {
            'yield(%)': 0.15, 'Nc(%)': 0.20, 'Sc(%)': 0.15,
            'Oc(%)': 0.20, 'HC- Ash(%)': 0.20, 'HHVc (MJ/Kg)': 0.10
        },
        'targets': {
            'yield(%)': 'max', 'Nc(%)': 'max', 'Sc(%)': 'max',
            'Oc(%)': 'max', 'HC- Ash(%)': 'max', 'HHVc (MJ/Kg)': 'min'
        }
    }
}

for app_name, criteria in application_criteria.items():
    print(f"\n🔹 {app_name}: {criteria['name']}")
    print(f"   Weights: {', '.join([f'{k}={v}' for k, v in criteria['weights'].items() if v > 0])}")


📊 Application criteria

🔹 Energy: Energy Production
   Weights: yield(%)=0.15, Cc (%)=0.1, Hc(%)=0.05, Nc(%)=0.05, Sc(%)=0.05, Oc(%)=0.05, HHVc (MJ/Kg)=0.25, EY(%)=0.05, ED(%)=0.05, CR(%)=0.05, HC- Volatile matter%=0.05, HC- Fixed carbon%=0.1, HC- Ash(%)=0.05

🔹 Soil: Soil Amendment
   Weights: yield(%)=0.15, Nc(%)=0.25, Sc(%)=0.2, HC- Ash(%)=0.3, HHVc (MJ/Kg)=0.1

🔹 Adsorbent: Adsorbent Material
   Weights: yield(%)=0.15, Nc(%)=0.2, Sc(%)=0.15, Oc(%)=0.2, HC- Ash(%)=0.2, HHVc (MJ/Kg)=0.1


# Step 5: Define NSGA-II problem

In [6]:
# ============================================================
# Step 5: Define NSGA-II problem
# ============================================================

print("\n" + "=" * 70)
print("🔄 Defining NSGA-II problem...")
print("=" * 70)

class OptimizationProblem(Problem):
    def __init__(self, raw_material, models_forward, input_cols, output_cols, 
                 weights, targets, df, output_mins, output_maxs):
        
        self.raw_material = raw_material
        self.models_forward = models_forward
        self.output_cols = output_cols
        self.weights = weights
        self.targets = targets
        self.output_mins = output_mins
        self.output_maxs = output_maxs
        
        # Temperature and time bounds
        temp_min = df['Temprature(°C)'].min()
        temp_max = df['Temprature(°C)'].max()
        time_min = df['Reaction Time (min)'].min()
        time_max = df['Reaction Time (min)'].max()
        
        xl = np.array([temp_min, time_min])
        xu = np.array([temp_max, time_max])
        
        # Active objectives (with weight > 0)
        self.active_outputs = [col for col, w in weights.items() if w > 0]
        n_obj = len(self.active_outputs)
        
        super().__init__(
            n_var=2,
            n_obj=n_obj,
            xl=xl,
            xu=xu,
            elementwise=False
        )
    
    def _evaluate(self, X, out, *args, **kwargs):
        n_solutions = X.shape[0]
        
        # Repeat raw material for each solution
        X_input = np.tile(self.raw_material, (n_solutions, 1))
        X_input = np.column_stack([X_input, X[:, 0], X[:, 1]])
        
        # Predict all outputs
        predictions = {}
        for col in self.output_cols:
            predictions[col] = self.models_forward[col].predict(X_input)
        
        # Calculate objectives
        F = np.zeros((n_solutions, self.n_obj))
        
        for i, output_col in enumerate(self.active_outputs):
            pred = predictions[output_col]
            min_val = self.output_mins[output_col]
            max_val = self.output_maxs[output_col]
            weight = self.weights[output_col]
            
            # Normalize
            if max_val > min_val:
                norm_val = (pred - min_val) / (max_val - min_val + 1e-10)
            else:
                norm_val = 0.5
            
            # Maximize or minimize
            if self.targets.get(output_col) == 'min':
                F[:, i] = norm_val * weight
            else:
                F[:, i] = -norm_val * weight
        
        out["F"] = F

print("✅ NSGA-II problem defined successfully")


🔄 Defining NSGA-II problem...
✅ NSGA-II problem defined successfully


# 🚀 # Step 6: Generate optimal data (271 points × 3 applications)

In [7]:
# ============================================================
# Step 6: Generate optimal data (271 points × 3 applications) - IMPROVED
# ============================================================

print("\n" + "=" * 70)
print("🔄 Generating optimal data with improved ranges...")
print("=" * 70)

# ============================================================
# 🔥 NEW OPTIMIZATION PROBLEM WITH IMPROVED RANGES
# ============================================================

class OptimizationProblemImproved(Problem):
    def __init__(self, raw_material, models_forward, input_cols, output_cols, 
                 weights, targets, df, output_mins, output_maxs):
        
        self.raw_material = raw_material
        self.models_forward = models_forward
        self.output_cols = output_cols
        self.weights = weights
        self.targets = targets
        self.output_mins = output_mins
        self.output_maxs = output_maxs
        
        # ============================================================
        # 🔥 IMPROVED RANGES (Based on data analysis)
        # ============================================================
        
        # Temperature: Natural range (5%-95%)
        temp_min = 170   # 5th percentile
        temp_max = 260   # 95th percentile
        
        # Time: Improved range (avoid outliers)
        time_min = 30    # Minimum
        time_max = 180   # Maximum (avoid values > 180 min)
        
        # ============================================================
        
        xl = np.array([temp_min, time_min])
        xu = np.array([temp_max, time_max])
        
        self.active_outputs = [col for col, w in weights.items() if w > 0]
        n_obj = len(self.active_outputs)
        
        super().__init__(
            n_var=2,
            n_obj=n_obj,
            xl=xl,
            xu=xu,
            elementwise=False
        )
    
    def _evaluate(self, X, out, *args, **kwargs):
        n_solutions = X.shape[0]
        
        X_input = np.tile(self.raw_material, (n_solutions, 1))
        X_input = np.column_stack([X_input, X[:, 0], X[:, 1]])
        
        predictions = {}
        for col in self.output_cols:
            predictions[col] = self.models_forward[col].predict(X_input)
        
        F = np.zeros((n_solutions, self.n_obj))
        
        for i, output_col in enumerate(self.active_outputs):
            pred = predictions[output_col]
            min_val = self.output_mins[output_col]
            max_val = self.output_maxs[output_col]
            weight = self.weights[output_col]
            
            if max_val > min_val:
                norm_val = (pred - min_val) / (max_val - min_val + 1e-10)
            else:
                norm_val = 0.5
            
            if self.targets.get(output_col) == 'min':
                F[:, i] = norm_val * weight
            else:
                F[:, i] = -norm_val * weight
        
        out["F"] = F

# ============================================================
# GENERATE OPTIMAL DATA WITH IMPROVED RANGES
# ============================================================

def generate_optimal_data_improved(df, models_forward, input_cols, output_cols,
                                   output_mins, output_maxs, application_criteria):
    
    all_data = []
    
    for app_name, criteria in application_criteria.items():
        print(f"\n📊 Generating data for application: {app_name}")
        print(f"   Number of points: {len(df)}")
        print(f"   Temperature range: 170-260°C")
        print(f"   Time range: 30-180 min")
        
        weights = criteria['weights']
        targets = criteria['targets']
        
        for idx in range(len(df)):
            raw_material = [df.iloc[idx][col] for col in base_input_cols]
            
            # ============================================================
            # USING IMPROVED PROBLEM WITH BETTER RANGES
            # ============================================================
            
            problem = OptimizationProblemImproved(
                raw_material, models_forward, input_cols, output_cols,
                weights, targets, df, output_mins, output_maxs
            )
            
            algorithm = NSGA2(
                pop_size=20,
                sampling=FloatRandomSampling(),
                crossover=SBX(prob=0.9, eta=15),
                mutation=PM(prob=0.1, eta=20),
                eliminate_duplicates=True
            )
            
            res = minimize(
                problem,
                algorithm,
                ('n_gen', 10),
                seed=42 + idx,
                verbose=False
            )
            
            if len(res.X) > 0:
                best_x = res.X[np.argmin(np.sum(res.F, axis=1))]
                temp, time = best_x
                
                # Predict outputs
                X_input = np.array(raw_material + [temp, time]).reshape(1, -1)
                
                predictions = {}
                for col in output_cols:
                    predictions[col] = models_forward[col].predict(X_input)[0]
                
                # Save data
                row = {'Application': app_name, 'Temperature': temp, 'Time': time}
                
                for i, col in enumerate(base_input_cols):
                    row[col] = raw_material[i]
                
                for col in output_cols:
                    row[f'Pred_{col}'] = predictions[col]
                
                all_data.append(row)
            
            if (idx + 1) % 50 == 0:
                print(f"   Generated {idx+1}/{len(df)} points")
    
    return pd.DataFrame(all_data)

# ============================================================
# GENERATE DATA
# ============================================================

optimal_data_improved = generate_optimal_data_improved(
    df, models_forward, input_cols, output_cols,
    output_mins, output_maxs, application_criteria
)

print(f"\n✅ Generated {len(optimal_data_improved)} optimal data points (improved ranges)")
print(f"   Distribution:")
for app in ['Energy', 'Soil', 'Adsorbent']:
    count = len(optimal_data_improved[optimal_data_improved['Application'] == app])
    print(f"   - {app}: {count} points")

# ============================================================
# SAVE DATA
# ============================================================

optimal_data_improved.to_excel('/kaggle/working/optimal_data_813_improved.xlsx', index=False)
print("✅ Data saved to 'optimal_data_813_improved.xlsx'")

# ============================================================
# QUICK VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("📊 Quick validation of improved data")
print("=" * 70)

print(f"\n📊 Time statistics:")
print(f"   Min: {optimal_data_improved['Time'].min():.1f} min")
print(f"   Max: {optimal_data_improved['Time'].max():.1f} min")
print(f"   Mean: {optimal_data_improved['Time'].mean():.1f} min")

print(f"\n📊 Temperature statistics:")
print(f"   Min: {optimal_data_improved['Temperature'].min():.1f} °C")
print(f"   Max: {optimal_data_improved['Temperature'].max():.1f} °C")
print(f"   Mean: {optimal_data_improved['Temperature'].mean():.1f} °C")

# Check for outliers
time_outliers = optimal_data_improved[optimal_data_improved['Time'] > 200]
temp_outliers = optimal_data_improved[optimal_data_improved['Temperature'] > 280]

if len(time_outliers) == 0 and len(temp_outliers) == 0:
    print("\n✅ ✅ ✅ All values are within the improved ranges!")
    print("   No outliers detected in the generated data")
else:
    print(f"\n⚠️ Found {len(time_outliers)} time outliers")
    print(f"⚠️ Found {len(temp_outliers)} temperature outliers")

print("\n" + "=" * 70)
print("✅ Generation complete with improved ranges!")
print("=" * 70)


🔄 Generating optimal data with improved ranges...

📊 Generating data for application: Energy
   Number of points: 271
   Temperature range: 170-260°C
   Time range: 30-180 min
   Generated 50/271 points
   Generated 100/271 points
   Generated 150/271 points
   Generated 200/271 points
   Generated 250/271 points

📊 Generating data for application: Soil
   Number of points: 271
   Temperature range: 170-260°C
   Time range: 30-180 min
   Generated 50/271 points
   Generated 100/271 points
   Generated 150/271 points
   Generated 200/271 points
   Generated 250/271 points

📊 Generating data for application: Adsorbent
   Number of points: 271
   Temperature range: 170-260°C
   Time range: 30-180 min
   Generated 50/271 points
   Generated 100/271 points
   Generated 150/271 points
   Generated 200/271 points
   Generated 250/271 points

✅ Generated 813 optimal data points (improved ranges)
   Distribution:
   - Energy: 271 points
   - Soil: 271 points
   - Adsorbent: 271 points
✅ Data s

# Step 7: Train Inverse Model

In [8]:
# ============================================================
# Step 7: Train Inverse Model (ON IMPROVED DATA)
# ============================================================

print("\n" + "=" * 70)
print("🔄 Training inverse model on improved data...")
print("=" * 70)

# ============================================================
# 🔥 تحميل البيانات الجديدة (المحسّنة)
# ============================================================

optimal_data = pd.read_excel('/kaggle/working/optimal_data_813_improved.xlsx')
print(f"✅ Loaded improved data: {len(optimal_data)} points")

# ============================================================

# Add One-Hot Encoding for applications
applications = ['Energy', 'Soil', 'Adsorbent']
for app in applications:
    optimal_data[f'app_{app}'] = (optimal_data['Application'] == app).astype(int)

# Inputs: feedstock properties (11) + application (3) = 14
inverse_input_cols = base_input_cols + [f'app_{app}' for app in applications]

# Outputs: temperature, time, and product properties (13+2 = 15)
inverse_output_cols = ['Temperature', 'Time'] + [f'Pred_{col}' for col in output_cols]

X_inv = optimal_data[inverse_input_cols]
Y_inv = optimal_data[inverse_output_cols]

# Split data
X_inv_train, X_inv_test, Y_inv_train, Y_inv_test = train_test_split(
    X_inv, Y_inv, test_size=0.2, random_state=42
)

print(f"✅ Training data: {len(X_inv_train)} points")
print(f"✅ Testing data: {len(X_inv_test)} points")

# Train inverse model
inverse_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

inverse_model.fit(X_inv_train, Y_inv_train)
print("✅ Inverse model trained successfully on improved data")

# ============================================================
# 💾 حفظ النموذج الجديد (باسم مختلف)
# ============================================================

joblib.dump(inverse_model, '/kaggle/working/inverse_model_improved.pkl')
print("✅ Model saved to 'inverse_model_improved.pkl'")


🔄 Training inverse model on improved data...
✅ Loaded improved data: 813 points
✅ Training data: 650 points
✅ Testing data: 163 points
✅ Inverse model trained successfully on improved data
✅ Model saved to 'inverse_model_improved.pkl'


# 📊 # Step 8: Evaluate inverse model

In [9]:
# ============================================================
# Step 8: Evaluate inverse model
# ============================================================

print("\n" + "=" * 70)
print("📊 Evaluating inverse model")
print("=" * 70)

Y_inv_pred = inverse_model.predict(X_inv_test)

print("\n📈 R² for each output:")
print("-" * 50)

metrics = []
for i, col in enumerate(inverse_output_cols):
    r2 = r2_score(Y_inv_test[col], Y_inv_pred[:, i])
    rmse = np.sqrt(mean_squared_error(Y_inv_test[col], Y_inv_pred[:, i]))
    
    status = "✅ Excellent" if r2 >= 0.7 else "🟡 Good" if r2 >= 0.5 else "🔴 Poor"
    print(f"   {col:30} R² = {r2:.4f}  {status}")
    metrics.append({'Output': col, 'R²': r2, 'RMSE': rmse})

avg_r2 = np.mean([m['R²'] for m in metrics])
print(f"\n📊 Average R²: {avg_r2:.4f}")

# Save model
joblib.dump(inverse_model, '/kaggle/working/inverse_model.pkl')
print("✅ Inverse model saved to 'inverse_model.pkl'")


📊 Evaluating inverse model

📈 R² for each output:
--------------------------------------------------
   Temperature                    R² = 0.8272  ✅ Excellent
   Time                           R² = 0.6527  🟡 Good
   Pred_yield(%)                  R² = 0.9108  ✅ Excellent
   Pred_Cc (%)                    R² = 0.9806  ✅ Excellent
   Pred_Hc(%)                     R² = 0.9747  ✅ Excellent
   Pred_Nc(%)                     R² = 0.9024  ✅ Excellent
   Pred_Sc(%)                     R² = 0.9387  ✅ Excellent
   Pred_Oc(%)                     R² = 0.9411  ✅ Excellent
   Pred_HHVc (MJ/Kg)              R² = 0.9812  ✅ Excellent
   Pred_EY(%)                     R² = 0.8627  ✅ Excellent
   Pred_ED(%)                     R² = 0.9365  ✅ Excellent
   Pred_CR(%)                     R² = 0.8860  ✅ Excellent
   Pred_HC- Volatile matter%      R² = 0.9156  ✅ Excellent
   Pred_HC- Fixed carbon%         R² = 0.9082  ✅ Excellent
   Pred_HC- Ash(%)                R² = 0.9889  ✅ Excellent

📊 Average R²: 0.9

# 🖥️ # Step 9: Interactive Dashboard

In [10]:
# ============================================================
# 🌱 Hydrochar from Digestat AI System - MODERN DESIGN
# ============================================================

!pip install gradio openpyxl -q

import gradio as gr
import numpy as np
import pandas as pd
import joblib
import datetime

# ============================================================
# 🔧 TRY TO IMPORT SPACES (only works on Hugging Face)
# ============================================================

try:
    import spaces
    SPACES_AVAILABLE = True
    print("✅ Running on Hugging Face Spaces with GPU support")
except ImportError:
    SPACES_AVAILABLE = False
    print("ℹ️ Running locally (Kaggle/Colab) - GPU not available")
    
    # Create a dummy decorator if spaces not available
    class DummySpaces:
        @staticmethod
        def GPU(func):
            return func
    
    spaces = DummySpaces()

# ============================================================
# LOAD MODEL
# ============================================================

try:
    model = joblib.load("/kaggle/working/inverse_model.pkl")
    print("✅ Model loaded")
except:
    print("⚠️ Demo model loaded")
    class DummyModel:
        def predict(self, X):
            return np.array([[
                220, 180, 72, 68, 5, 1.8, 0.4, 22, 28, 75, 35, 80, 40, 50, 8
            ]])
    model = DummyModel()

# ============================================================
# OUTPUT NAMES
# ============================================================

output_names = [
    "Temperature (°C)",
    "Residence Time (min)",
    "Yield (%)",
    "Cc (%)",
    "Hc (%)",
    "Nc (%)",
    "Sc (%)",
    "Oc (%)",
    "HHVc (MJ/Kg)",
    "EY (%)",
    "ED (%)",
    "CR (%)",
    "HC- Volatile matter (%)",
    "HC- Fixed carbon (%)",
    "HC- Ash (%)"
]

# ============================================================
# PREDICTION FUNCTION WITH GPU SUPPORT (if available)
# ============================================================

@spaces.GPU  # This works on Hugging Face, ignored locally
def prediction(
    C, H, N, S, O,
    Ash, VM, FC, HHV,
    DM, Water, Application
):
    app = [
        1 if Application == "Energy" else 0,
        1 if Application == "Soil" else 0,
        1 if Application == "Adsorbent" else 0
    ]
    
    X = np.array([
        C, H, N, S, O,
        Ash, VM, FC, HHV,
        DM, Water, *app
    ]).reshape(1, -1)
    
    y = model.predict(X)[0]
    
    # HTML مع عرض في صفين
    html = """
    <style>
        .result-container {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            padding: 12px 15px;
            border-radius: 12px;
            box-shadow: 0 8px 30px rgba(102, 126, 234, 0.3);
            margin: 0px;
        }
        .result-title {
            color: white;
            font-size: 15px;
            font-weight: bold;
            text-align: center;
            margin-bottom: 10px;
            text-shadow: 0 2px 4px rgba(0,0,0,0.2);
            letter-spacing: 0.5px;
        }
        .result-grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 8px;
            background: rgba(255,255,255,0.95);
            border-radius: 10px;
            padding: 10px 12px;
            box-shadow: 0 4px 12px rgba(0,0,0,0.08);
        }
        .result-item {
            display: flex;
            justify-content: space-between;
            padding: 5px 8px;
            border-bottom: 1px solid #f0f0f0;
            transition: all 0.3s ease;
            border-radius: 4px;
        }
        .result-item:hover {
            background: linear-gradient(90deg, #f8f9fa 0%, #e9ecef 100%);
            transform: scale(1.02);
        }
        .result-item .property {
            font-size: 12px;
            color: #2d3748;
            font-weight: 500;
        }
        .result-item .value {
            font-size: 13px;
            font-weight: bold;
            color: #667eea;
        }
        .footer-note {
            text-align: center;
            color: rgba(255,255,255,0.8);
            font-size: 11px;
            margin-top: 8px;
        }
        .result-item:last-child {
            border-bottom: none;
        }
        .result-item:nth-last-child(2) {
            border-bottom: none;
        }
    </style>
    
    <div class="result-container">
        <div class="result-title">
            📊 Hydrochar Properties
        </div>
        <div class="result-grid">
    """
    
    # عرض الخصائص في صفين (عمودين)
    for i, (p, v) in enumerate(zip(output_names, y[:15])):
        html += f"""
            <div class="result-item">
                <span class="property">{p}</span>
                <span class="value">{v:.3f}</span>
            </div>
        """
    
    html += """
        </div>
        <div class="footer-note">
            ⚡ Powered by Random Forest AI Model
        </div>
    </div>
    """
    
    return html

# ============================================================
# CSS - DESIGN MODERN
# ============================================================

css = """
.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
    background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%) !important;
    padding: 20px !important;
    border-radius: 20px !important;
    border: 4px solid #667eea !important;
    box-shadow: 0 0 0 8px #764ba2, 0 20px 60px rgba(0,0,0,0.15) !important;
}

.header {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
    padding: 10px 20px !important;
    border-radius: 10px !important;
    text-align: center !important;
    margin-bottom: 0px !important;
    box-shadow: 0 6px 20px rgba(102, 126, 234, 0.2) !important;
    border: none !important;
}

.header h1 {
    font-size: 18px !important;
    color: white !important;
    margin: 0 !important;
    font-weight: 600 !important;
    text-shadow: 0 2px 4px rgba(0,0,0,0.15) !important;
    letter-spacing: 0.5px !important;
}

.box {
    background: rgba(255,255,255,0.95) !important;
    padding: 12px !important;
    border-radius: 12px !important;
    border: 2px solid #e2e8f0 !important;
    box-shadow: 0 8px 32px rgba(0,0,0,0.08) !important;
    margin-top: 0px !important;
    transition: all 0.3s ease !important;
}

.box:hover {
    box-shadow: 0 12px 40px rgba(0,0,0,0.12) !important;
    transform: translateY(-2px) !important;
}

.box h3, .box .gr-markdown {
    display: none !important;
}

input, .gr-number input, .gr-textbox input {
    height: 32px !important;
    font-size: 11px !important;
    border: 2px solid #e2e8f0 !important;
    border-radius: 8px !important;
    background: #f7fafc !important;
}

input:focus, .gr-number input:focus {
    border-color: #667eea !important;
    box-shadow: 0 0 0 3px rgba(102, 126, 234, 0.2) !important;
    background: white !important;
    outline: none !important;
}

select, .gr-dropdown select {
    height: 32px !important;
    font-size: 11px !important;
    border: 2px solid #e2e8f0 !important;
    border-radius: 8px !important;
    background: linear-gradient(135deg, #f7fafc 0%, #edf2f7 100%) !important;
}

select:focus {
    border-color: #667eea !important;
    box-shadow: 0 0 0 3px rgba(102, 126, 234, 0.2) !important;
}

button {
    height: 36px !important;
    font-size: 14px !important;
    font-weight: 600 !important;
    border: none !important;
    border-radius: 10px !important;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
    color: white !important;
    transition: all 0.3s ease !important;
    box-shadow: 0 4px 15px rgba(102, 126, 234, 0.3) !important;
    letter-spacing: 0.5px !important;
}

button:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 25px rgba(102, 126, 234, 0.4) !important;
}

.tabs {
    display: none !important;
}

label, .gr-label {
    font-weight: 500 !important;
    color: #2d3748 !important;
    font-size: 11px !important;
    margin-bottom: 2px !important;
}

.gr-file, .download-btn, .file-download {
    display: none !important;
}

.gr-row {
    gap: 12px !important;
}

.gr-box {
    margin-top: 0px !important;
    padding-top: 0px !important;
}

.gr-row .gr-column {
    padding-top: 0px !important;
}

.gr-row .gr-column .gr-row {
    gap: 6px !important;
}

.gr-row .gr-column .gr-column {
    padding: 0px 4px !important;
}

.footer, .gradio-footer, .gradio-footer-version,
.built-with, .built-with-gradio,
.app-footer, .footer-text,
.parameters, .api-parameters,
.utilizer, .utilizer-text,
[class*="utilizer"], [class*="Parameters"],
[class*="api"], [class*="API"] {
    display: none !important;
}

div:has(> span:contains("Built with Gradio")),
div:has(> span:contains("utlizer")),
div:has(> span:contains("Parameters")),
div:has(> span:contains("API")) {
    display: none !important;
}

.dashboard-frame {
    border: 3px solid #667eea !important;
    border-radius: 20px !important;
    padding: 5px !important;
    background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%) !important;
    box-shadow: 0 0 30px rgba(102, 126, 234, 0.3) !important;
}
"""

# ============================================================
# INTERFACE
# ============================================================

with gr.Blocks(
    css=css,
    title="🌱 Hydrochar from Digestat",
    elem_classes="dashboard-frame"
) as demo:
    
    gr.HTML("""
    <div class="header">
        <h1>🌱 Hydrochar from Digestat</h1>
    </div>
    """)
    
    with gr.Row():
        with gr.Column(scale=1, elem_classes="box"):
            with gr.Row():
                with gr.Column(scale=1, min_width=60):
                    C = gr.Number(label="🟢 Carbon C (%)", value=45)
                    H = gr.Number(label="🔵 Hydrogen H (%)", value=6)
                    N = gr.Number(label="🟡 Nitrogen N (%)", value=1.2)
                with gr.Column(scale=1, min_width=60):
                    S = gr.Number(label="🟠 Sulfur S (%)", value=0.5)
                    O = gr.Number(label="🔴 Oxygen O (%)", value=38)
                    Ash = gr.Number(label="⚪ Ash (%)", value=8)
                with gr.Column(scale=1, min_width=60):
                    VM = gr.Number(label="🔥 Volatile Matter (%)", value=50)
                    FC = gr.Number(label="⬛ Fixed Carbon (%)", value=40)
                    HHV = gr.Number(label="⚡ HHV (MJ/kg)", value=20)
                with gr.Column(scale=1, min_width=60):
                    DM = gr.Number(label="💧 Dry Matter (%)", value=90)
                    Water = gr.Number(label="💦 Water (%)", value=10)
                    Application = gr.Dropdown(
                        ["Energy", "Soil", "Adsorbent"],
                        value="Energy",
                        label="🎯 Application"
                    )
            
            predict_btn = gr.Button("🚀 Generate Prediction", variant="primary")
        
        with gr.Column(scale=1, elem_classes="box"):
            output = gr.HTML()
    
    predict_btn.click(
        prediction,
        inputs=[C, H, N, S, O, Ash, VM, FC, HHV, DM, Water, Application],
        outputs=[output]
    )

# ============================================================
# RUN
# ============================================================

demo.launch(share=True, debug=False)

# ============================================================
# ℹ️ بعد تشغيل الكود، خذ الرابط العام وافتحه في المتصفح
# ============================================================

print("\n" + "="*60)
print("📸 TO CAPTURE SCREENSHOT:")
print("="*60)
print("1. Wait for the Gradio link to appear above")
print("2. Copy the public URL (starts with https://xxxxx.gradio.live)")
print("3. Open it in your browser")
print("4. Take screenshot using browser tools (F12 → Screenshot)")
print("="*60)

ℹ️ Running locally (Kaggle/Colab) - GPU not available
✅ Model loaded
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://2a75509182c2929288.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



📸 TO CAPTURE SCREENSHOT:
1. Wait for the Gradio link to appear above
2. Copy the public URL (starts with https://xxxxx.gradio.live)
3. Open it in your browser
4. Take screenshot using browser tools (F12 → Screenshot)
